# Lexicon-Based vs. Machine Learning Approaches for Financial Sentiment Classification

Runs all four approaches (VADER, Loughran-McDonald, TF-IDF+ML, FinBERT zero-shot) on the same held-out Financial PhraseBank test split and produces the comparison table/figures used in the writeup.

In [1]:
import sys
from pathlib import Path

SRC_PATH = Path.cwd().parent / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import nltk
for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    try:
        nltk.data.find(f"tokenizers/{resource}")
    except LookupError:
        try:
            nltk.data.find(f"corpora/{resource}")
        except LookupError:
            nltk.download(resource, quiet=True)

## 1. Load data and create the shared train/val/test split

In [2]:
from finsent.data.loader import load_financial_phrasebank
from finsent.data.splits import get_splits

df = load_financial_phrasebank()
splits = get_splits(df)

for name, split_df in splits.items():
    print(name, len(split_df), split_df["label"].value_counts().to_dict())

train 3392 {'neutral': 2015, 'positive': 954, 'negative': 423}
val 727 {'neutral': 432, 'positive': 205, 'negative': 90}
test 727 {'neutral': 432, 'positive': 204, 'negative': 91}


In [3]:
X_train, y_train = splits["train"]["sentence"].tolist(), splits["train"]["label"].tolist()
X_val, y_val = splits["val"]["sentence"].tolist(), splits["val"]["label"].tolist()
X_test, y_test = splits["test"]["sentence"].tolist(), splits["test"]["label"].tolist()

## 2. Instantiate the four models

All four implement the same `predict(texts) -> list[str]` interface (`finsent.models.base.SentimentModel`).

In [4]:
from finsent.models.vader_model import VaderModel
from finsent.models.loughran_mcdonald import LoughranMcDonaldModel
from finsent.models.tfidf_ml import TfidfMlModel
from finsent.models.finbert_model import FinBertModel

vader = VaderModel()
lm = LoughranMcDonaldModel()
tfidf_ml = TfidfMlModel()
finbert = FinBertModel()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## 3. Run the evaluation harness

VADER, Loughran-McDonald, and FinBERT need no training (`fit` is a no-op); TF-IDF+ML is fit on the train split with the classifier/`C` selected by val-set macro-F1.

In [5]:
from finsent.evaluation.metrics import compute_metrics

results = {}
predictions = {}

vader.fit(X_train, y_train)
predictions[vader.name] = vader.predict(X_test)
results[vader.name] = compute_metrics(y_test, predictions[vader.name])

lm.fit(X_train, y_train)
predictions[lm.name] = lm.predict(X_test)
results[lm.name] = compute_metrics(y_test, predictions[lm.name])

tfidf_ml.fit(X_train, y_train, val_texts=X_val, val_labels=y_val)
print("TF-IDF+ML selected config:", tfidf_ml.selected_config)
predictions[tfidf_ml.name] = tfidf_ml.predict(X_test)
results[tfidf_ml.name] = compute_metrics(y_test, predictions[tfidf_ml.name])
tfidf_ml.save()

finbert.fit(X_train, y_train)
predictions[finbert.name] = finbert.predict(X_test)
results[finbert.name] = compute_metrics(y_test, predictions[finbert.name])

{name: {"accuracy": r["accuracy"], "macro_f1": r["macro_f1"]} for name, r in results.items()}

TF-IDF+ML selected config: linear_svc(C=1.0)


{'VADER': {'accuracy': 0.53232462173315, 'macro_f1': 0.46890953675080516},
 'Loughran-McDonald': {'accuracy': 0.6327372764786795,
  'macro_f1': 0.5176694477268269},
 'TFIDF+ML': {'accuracy': 0.71939477303989, 'macro_f1': 0.6564080441379921},
 'FinBERT (zero-shot)': {'accuracy': 0.8775790921595599,
  'macro_f1': 0.8649890485789332}}

## 4. Generate the comparison table and figures

In [6]:
from finsent.evaluation.report import generate_full_report

comparison_df = generate_full_report(results)
comparison_df

,model,accuracy,macro_precision,macro_recall,macro_f1,neutral_f1
0,VADER,0.532325,0.495008,0.484426,0.468910,0.615385
1,Loughran-McDonald,0.632737,0.570508,0.514880,0.517669,0.750253
2,TFIDF+ML,0.719395,0.676389,0.641510,0.656408,0.802676
3,FinBERT (zero-shot),0.877579,0.838016,0.905405,0.864989,0.897059


## 7. Discussion

_Fill in after reviewing `results/tables/comparison.csv`, `significance.csv`, `hardest_sentences.csv`, and the confusion matrices in `results/figures/`:_
- How does the general-purpose VADER lexicon compare to the finance-specific Loughran-McDonald lexicon?
- Does the supervised TF-IDF+ML model outperform both lexicon methods, and is the difference statistically significant (see the McNemar's test results above)?
- How does FinBERT (zero-shot reference) compare, and does it particularly help on the neutral class?
- Where does the neutral class remain hardest across all four methods, and why (hedged/non-committal language)? Look at `neutral_errors_*.csv` and `hardest_sentences.csv` for concrete examples.
- Overall: when do simple, interpretable lexicon methods suffice vs. when is supervised learning warranted?

In [7]:
from finsent.evaluation.significance import pairwise_significance

significance_df = pairwise_significance(y_test, predictions)
significance_df

,model_a,model_b,n_both_correct,n_a_only_correct,n_b_only_correct,n_both_wrong,p_value,significant
0,VADER,Loughran-McDonald,266,121,194,146,4.628192e-05,True
1,VADER,TFIDF+ML,280,107,243,97,2.693405e-13,True
2,VADER,FinBERT (zero-shot),352,35,286,54,4.023131e-50,True
3,Loughran-McDonald,TFIDF+ML,356,104,167,100,1.557004e-04,True
4,Loughran-McDonald,FinBERT (zero-shot),411,49,227,40,1.485804e-28,True
5,TFIDF+ML,FinBERT (zero-shot),482,41,156,48,5.376365e-17,True


## 6. Error analysis

Per-model misclassifications (with a neutral-class-specific slice), plus the sentences that trip up multiple models at once — the qualitative evidence for *why* lexicon methods struggle with hedged financial language. Saved to `results/tables/`.

In [8]:
from finsent.evaluation.error_analysis import generate_error_analysis

error_tables = generate_error_analysis(X_test, y_test, predictions)

for name in predictions:
    print(name, "-", len(error_tables[f"errors_{name}"]), "total errors,",
          len(error_tables[f"neutral_errors_{name}"]), "involve the neutral class")

print("\nSentences wrong on 2+ models:", len(error_tables["hardest_sentences"]))
error_tables["hardest_sentences"].head(10)

VADER - 340 total errors, 285 involve the neutral class
Loughran-McDonald - 267 total errors, 247 involve the neutral class
TFIDF+ML - 204 total errors, 177 involve the neutral class
FinBERT (zero-shot) - 89 total errors, 84 involve the neutral class

Sentences wrong on 2+ models: 255


,sentence,true_label,n_models_wrong,pred_VADER,pred_Loughran-McDonald,pred_TFIDF+ML,pred_FinBERT (zero-shot)
0,The company plans to spend the proceeds from t...,neutral,4,positive,positive,positive,positive
1,The company said that it will supply the WCDMA...,positive,4,neutral,neutral,neutral,neutral
2,A & euro ; 4.8 million investment in 13.6 % of...,positive,4,neutral,neutral,neutral,neutral
3,The last job losses related to these reduction...,neutral,4,negative,negative,negative,negative
4,Elisa will expand the use of this technology p...,neutral,4,positive,positive,positive,positive
5,A tinyurl link takes users to a scamming site ...,negative,4,positive,neutral,neutral,neutral
6,The company says it is difficult to estimate t...,neutral,4,negative,negative,negative,negative
7,"( ADP News ) - Nov 3 , 2008 - Finnish cargo ha...",positive,4,neutral,neutral,neutral,neutral
8,Sony Ericsson and Nokia dominated the list of ...,positive,4,neutral,neutral,neutral,neutral
9,The StoneGate UTM solution offers protection a...,positive,4,negative,neutral,neutral,neutral


## 5. Discussion

_Fill in after reviewing `results/tables/comparison.csv` and the confusion matrices in `results/figures/`:_
- How does the general-purpose VADER lexicon compare to the finance-specific Loughran-McDonald lexicon?
- Does the supervised TF-IDF+ML model outperform both lexicon methods, and by how much?
- How does FinBERT (zero-shot reference) compare, and does it particularly help on the neutral class?
- Where does the neutral class remain hardest across all four methods, and why (hedged/non-committal language)?
- Overall: when do simple, interpretable lexicon methods suffice vs. when is supervised learning warranted?